# SpMV CSC Accelerator  

In [ ]:
import time
import numpy as np
import scipy.sparse as sp
from pynq import Overlay, allocate

# =====================================================================
# 1. Initialization and IP Setup
# =====================================================================

# Load the overlay (Update the path to your actual .bit file)
bitstream_path = '/home/xilinx/spmv_csc/spmv_csc.bit'
print(f"Loading overlay from {bitstream_path}...")
overlay = Overlay(bitstream_path)
print('Overlay loaded!')

# Get handler of the spmv IP
spmv_ip = overlay.spmv_csc_0 

# =====================================================================
# 2. Problem Setup & Matrix Generation
# =====================================================================
print("\nRunning SPMV CSC testbench...")

num_rows = 1024
num_cols = 1024
density  = 0.05 # 5% non-zeros
PACK_SIZE = 16  # Hardware fetches 16 elements (512-bit) at a time

print(f"Generating random {num_rows}x{num_cols} sparse matrix (Density: {density})...")

np.random.seed(42)

# Generate Random CSC Matrix
sparse_matrix = sp.random(num_rows, num_cols, density=density, format='csc', dtype=np.float32, random_state=42)

# Extract CSC Arrays
A_values_data    = sparse_matrix.data
A_row_idx_data = sparse_matrix.indices.astype(np.int32)
A_col_ptr_data = sparse_matrix.indptr.astype(np.int32)

# Input vector 'x'
x_data = np.random.rand(num_cols).astype(np.float32)

nnz = sparse_matrix.nnz
print(f"Generated {nnz} Non-Zero (NNZ) elements.")

# =====================================================================
# 3. Buffer Allocation and Padding
# =====================================================================
# The hardware reads/writes in chunks of 16. 
# We must pad buffers to avoid out-of-bounds AXI reads.
padded_nnz = ((nnz + PACK_SIZE - 1) // PACK_SIZE) * PACK_SIZE
padded_cols = ((num_cols + PACK_SIZE - 1) // PACK_SIZE) * PACK_SIZE
padded_rows = ((num_rows + PACK_SIZE - 1) // PACK_SIZE) * PACK_SIZE
padded_col_ptr = ((len(A_col_ptr_data) + PACK_SIZE - 1) // PACK_SIZE) * PACK_SIZE

print(f"Allocating Contiguous Memory...")
A_val_buf     = allocate(shape=(padded_nnz,), dtype=np.float32)
A_row_idx_buf = allocate(shape=(padded_nnz,), dtype=np.int32)
A_col_ptr_buf = allocate(shape=(padded_col_ptr,), dtype=np.int32)
x_buf         = allocate(shape=(padded_cols,), dtype=np.float32)
y_buf         = allocate(shape=(padded_rows,), dtype=np.float32)

# Initialize Buffers with generated data
A_val_buf[:nnz] = A_values_data
A_row_idx_buf[:nnz] = A_row_idx_data
A_col_ptr_buf[:len(A_col_ptr_data)] = A_col_ptr_data
x_buf[:num_cols] = x_data

# Zero-out the padding lanes
A_val_buf[nnz:] = 0.0
A_row_idx_buf[nnz:] = 0
A_col_ptr_buf[len(A_col_ptr_data):] = 0
x_buf[num_cols:] = 0.0
y_buf[:] = 0.0

# Flush CPU cache to DDR
A_val_buf.sync_to_device()
A_row_idx_buf.sync_to_device()
A_col_ptr_buf.sync_to_device()
x_buf.sync_to_device()
y_buf.sync_to_device()

# =====================================================================
# 4. Hardware Execution
# =====================================================================
# Register Offsets based on provided header
ADDR_AP_CTRL   = 0x00
ADDR_NUM_ROWS  = 0x10
ADDR_NUM_COLS  = 0x18
ADDR_NNZ       = 0x20
ADDR_A_ROW_IDX = 0x28
ADDR_A_COL_PTR = 0x34
ADDR_A_VALUES  = 0x40
ADDR_X         = 0x4c
ADDR_Y         = 0x58

def write_64bit_address(ip, base_offset, address):
    """Writes a 64-bit physical address across two 32-bit registers."""
    ip.write(base_offset, address & 0xFFFFFFFF)
    ip.write(base_offset + 0x04, (address >> 32) & 0xFFFFFFFF)

print("\nConfiguring Registers...")
# Write Scalar Parameters
spmv_ip.write(ADDR_NUM_ROWS, num_rows)
spmv_ip.write(ADDR_NUM_COLS, num_cols)
spmv_ip.write(ADDR_NNZ, nnz)

# Write 64-bit Memory Pointers
write_64bit_address(spmv_ip, ADDR_A_ROW_IDX, A_row_idx_buf.device_address)
write_64bit_address(spmv_ip, ADDR_A_COL_PTR, A_col_ptr_buf.device_address)
write_64bit_address(spmv_ip, ADDR_A_VALUES, A_val_buf.device_address)
write_64bit_address(spmv_ip, ADDR_X, x_buf.device_address)
write_64bit_address(spmv_ip, ADDR_Y, y_buf.device_address)

print("Starting Hardware Accelerator...")
hw_start = time.time()

# Set ap_start
spmv_ip.write(ADDR_AP_CTRL, 0x01) 

# Poll ap_done (bit 1)
while (spmv_ip.read(ADDR_AP_CTRL) & 0x02) == 0:
    pass

hw_end = time.time()
print(f"HW Execution Time: {(hw_end - hw_start) * 1000:.4f} ms")

# Retrieve Results
y_buf.sync_from_device()
y_hw = np.array(y_buf[:num_rows])

# =====================================================================
# 5. Software Reference & Verification
# =====================================================================
print("\nRunning Software Reference...")

sw_start = time.time()

# Build a dtype-stable CSC matrix and compute y = A * x (matching HW float32).
A_values_np = np.asarray(A_values_data, dtype=np.float32)
A_row_idx_np = np.asarray(A_row_idx_data, dtype=np.int32)
A_col_ptr_np = np.asarray(A_col_ptr_data, dtype=np.int32)

A_csc = sp.csc_matrix(
    (A_values_np, A_row_idx_np, A_col_ptr_np),
    shape=(num_rows, num_cols),
    dtype=np.float32,
)
y_sw = A_csc.dot(x_data)
y_sw = np.asarray(y_sw, dtype=np.float32).ravel()

sw_end = time.time()
print(f"SW Execution Time: {(sw_end - sw_start) * 1000:.4f} ms")

# Verification
print("\n--- Results ---")

print("\nResult Vector (y) [Showing first 10 rows]:")
print("Row | SW Ref       | HW Accel")
print("----------------------------------")
for i in range(min(10, num_rows)):
    print(f"{i:3} | {y_sw[i]:10.4f} | {y_hw[i]:10.4f}")
print("...\n")

# Use numpy's allclose to check with a relative tolerance (matches your 0.05% margin logic)
# rtol=0.0005 is equivalent to your C++ logic: diff / expected > 0.0005f
is_match = np.allclose(y_hw, y_sw, rtol=0.0005, atol=1e-5)

if is_match:
    print(f">>> SUCCESS: All {num_rows} hardware results match software reference! <<<")
else:
    # Print out mismatch details
    diffs = np.abs(y_hw - y_sw)
    rel_errors = diffs / (np.abs(y_sw) + 1e-8)
    mismatches = np.where(rel_errors > 0.0005)[0]
    
    print(f">>> ERROR: {len(mismatches)} Mismatches detected! <<<")
    for i in mismatches[:10]:
        print(f"Mismatch at row {i}: HW = {y_hw[i]:.6f}, SW = {y_sw[i]:.6f} (Diff = {diffs[i]:.6f})")

# Free memory buffers
A_val_buf.freebuffer()
A_row_idx_buf.freebuffer()
A_col_ptr_buf.freebuffer()
x_buf.freebuffer()
y_buf.freebuffer()  